# S0 — 02.05: Ridge Regression

**Question:** How does adding an L2 penalty change the closed-form regression solution while leaving the prediction rule and data unchanged?

This lesson uses five synthetic, dimensionless observations. It demonstrates the mechanics of a regularised training objective; it does not demonstrate performance, generalisation, extrapolation, or deployment.

## S1 — Continuity from earlier regression lessons

02.02 introduced closed-form OLS, 02.03 separated the model, objective, and optimizer, and 02.04 changed the features. Here the affine prediction rule and data remain unchanged; the objective gains an L2 penalty.

In [ ]:
import numpy as np

RTOL = 0.0
ATOL = 1e-12
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 1.0], [1.0, 2.0]], dtype=np.float64)
y = np.array([5.0, 1.0, 4.0, 8.0, 3.0], dtype=np.float64)
A = np.column_stack((np.ones(X.shape[0], dtype=np.float64), X))
I_mod = np.diag([0.0, 1.0, 1.0]).astype(np.float64)
print('X.shape:', X.shape, 'y.shape:', y.shape)
print('A.shape:', A.shape, 'A.T @ A:\n', A.T @ A)
print('A.T @ y:', A.T @ y)

## S2 — Objective components

With `theta = [b, w1, w2]`, predictions are `A @ theta`. The sum-scale Ridge objective is `J = RSS + lambda * (w1² + w2²)`. `lambda` is the hyperparameter; the intercept is not penalised.

## S3 — Modified normal equations

The zero in `I_mod = diag(0, 1, 1)` protects the intercept. Differentiating the objective gives `(A.T @ A + lambda * I_mod) @ theta = A.T @ y`. At `lambda = 0`, the OLS system returns.

In [ ]:
def ridge_theta(X, y, lam):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    A = np.column_stack((np.ones(X.shape[0], dtype=np.float64), X))
    return np.asarray(np.linalg.solve(A.T @ A + lam * I_mod, A.T @ y), dtype=np.float64)

for lam in (0.0, 0.5, 5.0):
    theta = ridge_theta(X, y, lam)
    predictions = A @ theta
    residuals = y - predictions
    rss = np.float64(np.sum(residuals**2, dtype=np.float64))
    mse = np.float64(rss / y.size)
    penalty = np.float64(lam * np.sum(theta[1:]**2, dtype=np.float64))
    print(f'lambda={lam}: theta={theta}, RSS={rss}, MSE={mse}, penalty={penalty}, J={rss + penalty}')

## S4 — Frozen arithmetic

For `lambda = 0.5`, `theta = [2.2, 2.8, -0.8]`, predictions are `[5.0, 1.4, 4.2, 7.0, 3.4]`, RSS is `1.36`, MSE is `0.272`, penalty is `4.24`, and `J = 5.6`.

For `lambda = 5.0`, `theta = [3.4857142857142858, 1.0, -0.2857142857142857]`, RSS is `13.820408163265306`, MSE is `2.764081632653061`, penalty is `5.408163265306122`, and `J = 19.228571428571428`.

## S5 — Scale bridge and library comparison

Dividing the complete objective by `N` gives `MSE + (lambda/N) * ||w||²`. Under this sum-scale convention, `sklearn.linear_model.Ridge(alpha=lambda)` is the matching positive-penalty comparison. `LinearRegression` supplies the lambda-zero reference.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

ols = LinearRegression(fit_intercept=True).fit(X, y)
print('lambda=0:', np.r_[ols.intercept_, ols.coef_])
for lam in (0.5, 5.0):
    model = Ridge(alpha=lam, fit_intercept=True, solver='cholesky').fit(X, y)
    print(f'lambda={lam}:', np.r_[model.intercept_, model.coef_])

## S6 — Interpret cautiously

The coefficient paths and objective components below are mechanical facts about this five-row training table. Feature weights move toward zero while the unpenalised intercept rises toward `mean(y) = 4.2`; this is not a claim about unseen-data quality. The static visual is paired with its numeric text equivalent in the lesson records.

## S7 — Independent verification

The private cross-check rebuilds the augmented matrices, displays inverse calculations for verification only, and independently recomputes coefficients, predictions, residuals, RSS, penalty, and `J`. It does not import the primary implementation.

In [ ]:
for lam, expected_theta in {
    0.0: np.array([1.7, 3.5, -1.0], dtype=np.float64),
    0.5: np.array([2.2, 2.8, -0.8], dtype=np.float64),
    5.0: np.array([3.4857142857142858, 1.0, -0.2857142857142857], dtype=np.float64),
}.items():
    independent_theta = np.asarray(np.linalg.solve(A.T @ A + lam * I_mod, A.T @ y), dtype=np.float64)
    np.testing.assert_allclose(independent_theta, expected_theta, rtol=RTOL, atol=ATOL)
print('independent coefficient checks passed')

## S8 — Read the coefficient and objective paths

Across `lambda = [0, 0.5, 1, 2, 5, 10]`, feature weights move toward zero while the unpenalised intercept rises. On this dataset, RSS and `J` rise across the displayed grid, while the penalty value is non-monotone and peaks at `lambda = 2`. These are mechanical facts about the frozen training table, not performance or generalisation claims.

In [ ]:
lambdas = np.array([0.0, 0.5, 1.0, 2.0, 5.0, 10.0], dtype=np.float64)
path_theta = np.asarray([ridge_theta(X, y, lam) for lam in lambdas], dtype=np.float64)
print('lambda:', lambdas)
print('intercept:', path_theta[:, 0])
print('feature weights:', path_theta[:, 1:])

## S9 — Bound the interpretation

The equal shrinkage visible here follows from the centered features having equal orthogonal spread; it is not a general Ridge rule. Feature scaling, lambda selection, validation, statistical inference, deployment, and conditioning theory are outside this introductory lesson.

## S10 — Accessible visual

![Ridge coefficient paths and objective components](assets/ridge_paths.png)

**Text equivalent:** The six lambda values are `0`, `0.5`, `1`, `2`, `5`, and `10`. Intercepts are `[1.7, 2.2, 2.533333, 2.95, 3.485714, 3.783333]`; `w1` values are `[3.5, 2.8, 2.333333, 1.75, 1.0, 0.583333]`; and `w2` values are `[-1.0, -0.8, -0.666667, -0.5, -0.285714, -0.166667]`. The stacked bars show RSS, L2 penalty, and total `J` for the same synthetic training cases. Meaning does not depend on colour or interaction.

## S11 — Exercises and misconceptions

1. Identify the loss, penalty, hyperparameter, and parameter vector in the sum-scale objective.
2. Explain why the leading zero in `I_mod` leaves the intercept unpenalised.
3. Reproduce the `lambda = 0.5` coefficients, predictions, RSS, penalty, and `J` by hand.
4. Explain why MSE can increase while the Ridge objective is the quantity being minimised.

A larger lambda is not automatically better, equal coefficient shrinkage is not universal, and this five-row example does not establish new-data performance.

## S12 — Recap and handoff

Ridge regression keeps the affine prediction rule but solves a penalised objective. The intercept is excluded from `||w||²`, the sum-scale convention is explicit, and the result is checked independently. Lambda selection and standardisation workflows are deferred. Lasso is the next conceptual handoff because its L1 penalty can produce exact zero coefficients.